### Using OpenAI (GPT-5.4) for data annotation

In [11]:
from openai import OpenAI 
import json

In [9]:
one_shot = """
1	350	O
2	,	O
3	Wellesley	B-LOC
4	,	O
5	Massachusetts	B-LOC
6	02481	O
7	doing	O
8	business	O
9	as	O
10	"	O
11	Silicon	B-LOC
12	Valley	I-LOC
13	East	I-LOC
14	"	O
15	and	O
16	AKAMAI	B-ORG
17	TECHNOLOGIES	I-ORG
18	,	O
19	INC	O
20	.	O
21	("	O
22	Borrower	B-PER
23	"),	O
"""

one_shot = one_shot.strip().split('\n')
one_shot = [line.strip().split('\t') for line in one_shot]

example_input = [line[1]for line in one_shot]
example_output = [{"token": line[1], "label": line[2]} for line in one_shot] 


test_case = """1	Dated	O
2	March	O
3	31	O
4	,	O
5	2007	O
6	Thinkplus	B-ORG
7	Investments	I-ORG
8	Limited	I-ORG
9	(	O
10	as	O
11	the	O
12	Lender	B-PER
13	)	O
14	AND	O
15	Airland	B-ORG
16	International	I-ORG
17	Limited	I-ORG
18	Bizexpress	I-ORG
19	Limited	I-ORG
20	(	O
21	as	O
22	the	O
23	Borrower	B-PER
24	)	O
25	Loan	O
26	Agreement	O
27	Contents	O"""


test_case = test_case.strip().split('\n')
test_case = [line.strip().split('\t') for line in test_case]

test_input = [line[1] for line in test_case]
test_output = [{"token": line[1], "label": line[2]} for line in test_case] 

print(example_input)
print(example_output)
print(test_input)
print(test_output)

['350', ',', 'Wellesley', ',', 'Massachusetts', '02481', 'doing', 'business', 'as', '"', 'Silicon', 'Valley', 'East', '"', 'and', 'AKAMAI', 'TECHNOLOGIES', ',', 'INC', '.', '("', 'Borrower', '"),']
[{'token': '350', 'label': 'O'}, {'token': ',', 'label': 'O'}, {'token': 'Wellesley', 'label': 'B-LOC'}, {'token': ',', 'label': 'O'}, {'token': 'Massachusetts', 'label': 'B-LOC'}, {'token': '02481', 'label': 'O'}, {'token': 'doing', 'label': 'O'}, {'token': 'business', 'label': 'O'}, {'token': 'as', 'label': 'O'}, {'token': '"', 'label': 'O'}, {'token': 'Silicon', 'label': 'B-LOC'}, {'token': 'Valley', 'label': 'I-LOC'}, {'token': 'East', 'label': 'I-LOC'}, {'token': '"', 'label': 'O'}, {'token': 'and', 'label': 'O'}, {'token': 'AKAMAI', 'label': 'B-ORG'}, {'token': 'TECHNOLOGIES', 'label': 'I-ORG'}, {'token': ',', 'label': 'O'}, {'token': 'INC', 'label': 'O'}, {'token': '.', 'label': 'O'}, {'token': '("', 'label': 'O'}, {'token': 'Borrower', 'label': 'B-PER'}, {'token': '"),', 'label': 'O'

In [12]:
from openai import OpenAI
client = OpenAI()

response = client.responses.create(
    model="gpt-5.4",  
    input=(
    
    "You are a strict NER tagger.\n\n"

    "Task:\n"
    "Assign a BIO tag to EACH token.\n\n"

    "Allowed labels:\n"
    "B-PER, I-PER, B-ORG, I-ORG, B-LOC, I-LOC, O\n\n"

    "Rules:\n"
    "- EXACTLY one label per token\n"
    "- SAME number of labels as tokens\n"
    "- SAME order as tokens\n"
    "- Do not modify tokens\n"
    "- Output MUST be valid JSON\n"
    "- No explanations\n\n"

    "Example input:\n"
    + json.dumps(example_input) + "\n\n"

    "Example output:\n"
    + json.dumps(example_output) + "\n\n"

    "Now annotate this input:\n"
    + json.dumps(test_input)
    )
)

openai_response = response.output[0].content[0].text
print(openai_response)

[{"token":"Dated","label":"O"},{"token":"March","label":"O"},{"token":"31","label":"O"},{"token":",","label":"O"},{"token":"2007","label":"O"},{"token":"Thinkplus","label":"B-ORG"},{"token":"Investments","label":"I-ORG"},{"token":"Limited","label":"I-ORG"},{"token":"(","label":"O"},{"token":"as","label":"O"},{"token":"the","label":"O"},{"token":"Lender","label":"O"},{"token":")","label":"O"},{"token":"AND","label":"O"},{"token":"Airland","label":"B-ORG"},{"token":"International","label":"I-ORG"},{"token":"Limited","label":"I-ORG"},{"token":"Bizexpress","label":"B-ORG"},{"token":"Limited","label":"I-ORG"},{"token":"(","label":"O"},{"token":"as","label":"O"},{"token":"the","label":"O"},{"token":"Borrower","label":"O"},{"token":")","label":"O"},{"token":"Loan","label":"O"},{"token":"Agreement","label":"O"},{"token":"Contents","label":"O"}]


In [13]:
openai_labels = json.loads(openai_response)

assert len(openai_labels) == len(test_output), "Number of labels does not match number of tokens."

print(f"TOKEN\tTOKEN_MATCH\tPREDICTED_LABEL\tTRUE_LABEL\tLABEL_MATCH")
for pred, true in zip(openai_labels, test_output):
    token = true['token']
    token_match = pred['token'] == true['token']
    pred_label = pred['label']
    true_label = true['label']
    label_match = "✓" if pred_label == true_label else "✗"
    print(f"{token}\t{token_match}\t{pred_label}\t{true_label}\t{label_match}")


TOKEN	TOKEN_MATCH	PREDICTED_LABEL	TRUE_LABEL	LABEL_MATCH
Dated	True	O	O	✓
March	True	O	O	✓
31	True	O	O	✓
,	True	O	O	✓
2007	True	O	O	✓
Thinkplus	True	B-ORG	B-ORG	✓
Investments	True	I-ORG	I-ORG	✓
Limited	True	I-ORG	I-ORG	✓
(	True	O	O	✓
as	True	O	O	✓
the	True	O	O	✓
Lender	True	O	B-PER	✗
)	True	O	O	✓
AND	True	O	O	✓
Airland	True	B-ORG	B-ORG	✓
International	True	I-ORG	I-ORG	✓
Limited	True	I-ORG	I-ORG	✓
Bizexpress	True	B-ORG	I-ORG	✗
Limited	True	I-ORG	I-ORG	✓
(	True	O	O	✓
as	True	O	O	✓
the	True	O	O	✓
Borrower	True	O	B-PER	✗
)	True	O	O	✓
Loan	True	O	O	✓
Agreement	True	O	O	✓
Contents	True	O	O	✓
